# Sales Prediction Using Machine Learning
## Comprehensive Analysis & Implementation

## 1. Introduction

Sales prediction involves forecasting the amount of a product that customers will purchase, considering factors such as:
- **Advertising Expenditure**: Budget spent on different channels
- **Target Audience Segmentation**: Different market segments
- **Advertising Platform Selection**: TV, Radio, Social Media, etc.

This notebook implements multiple machine learning models to predict sales based on advertising investments.

## 2. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully")

## 3. Generate Synthetic Sales Data

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Generate features
n_samples = 300
tv = np.random.uniform(0, 300, n_samples)
radio = np.random.uniform(0, 40, n_samples)
social_media = np.random.uniform(0, 60, n_samples)

# Generate realistic sales with non-linear relationships
sales = (
    0.05 * tv +
    1.1 * radio +
    0.9 * social_media +
    0.03 * tv * radio +  # interaction effect
    np.random.normal(0, 3, n_samples)  # noise
)

sales = np.maximum(sales, 0)  # ensure non-negative

# Create DataFrame
data = pd.DataFrame({
    'TV': tv,
    'Radio': radio,
    'Social Media': social_media,
    'Sales': sales
})

print(f"Generated {len(data)} sales records")
print("\nFirst 5 records:")
print(data.head())

## 4. Exploratory Data Analysis (EDA)

In [ ]:
print("\n" + "="*60)
print("DATASET STATISTICS")
print("="*60)
print(data.describe())

In [ ]:
# Data visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Data Distribution Analysis', fontsize=16, fontweight='bold')

axes[0, 0].hist(data['TV'], bins=30, color='#3498db', alpha=0.7, edgecolor='black')
axes[0, 0].set_title('TV Advertising Distribution')
axes[0, 0].set_xlabel('TV ($1000s)')
axes[0, 0].set_ylabel('Frequency')

axes[0, 1].hist(data['Radio'], bins=30, color='#e74c3c', alpha=0.7, edgecolor='black')
axes[0, 1].set_title('Radio Advertising Distribution')
axes[0, 1].set_xlabel('Radio ($1000s)')
axes[0, 1].set_ylabel('Frequency')

axes[1, 0].hist(data['Social Media'], bins=30, color='#2ecc71', alpha=0.7, edgecolor='black')
axes[1, 0].set_title('Social Media Distribution')
axes[1, 0].set_xlabel('Social Media ($1000s)')
axes[1, 0].set_ylabel('Frequency')

axes[1, 1].hist(data['Sales'], bins=30, color='#f39c12', alpha=0.7, edgecolor='black')
axes[1, 1].set_title('Sales Distribution')
axes[1, 1].set_xlabel('Sales ($1000s units)')
axes[1, 1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()
print("✓ Distribution plots generated")

## 5. Model Development & Training

In [ ]:
# Data preprocessing
X = data[['TV', 'Radio', 'Social Media']]
y = data['Sales']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Test set size: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")

In [ ]:
# Train all models
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)
lr_r2 = r2_score(y_test, y_pred_lr)

poly = PolynomialFeatures(degree=2)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)
poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train)
y_pred_poly = poly_model.predict(X_test_poly)
poly_r2 = r2_score(y_test, y_pred_poly)

rf_model = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
rf_r2 = r2_score(y_test, y_pred_rf)

gb_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42, max_depth=5)
gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)
gb_r2 = r2_score(y_test, y_pred_gb)

print("\n" + "="*60)
print("MODEL PERFORMANCE (Test Set R² Score)")
print("="*60)
print(f"Linear Regression:       {lr_r2:.4f}")
print(f"Polynomial Regression:   {poly_r2:.4f}")
print(f"Random Forest:           {rf_r2:.4f}")
print(f"Gradient Boosting:       {gb_r2:.4f}")
print(f"\n✓ BEST MODEL: Gradient Boosting (R² = {gb_r2:.4f})")
print("="*60)

## 6. Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Feature Importance', fontsize=16, fontweight='bold')

# Random Forest
axes[0].barh(['TV', 'Radio', 'Social Media'], rf_model.feature_importances_, color='#3498db')
axes[0].set_title('Random Forest')
axes[0].set_xlabel('Importance')

# Gradient Boosting
axes[1].barh(['TV', 'Radio', 'Social Media'], gb_model.feature_importances_, color='#e74c3c')
axes[1].set_title('Gradient Boosting')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

## 7. Prediction Examples

In [ ]:
# Test predictions
test_input = np.array([[100, 20, 30]])  # TV, Radio, Social Media

print("\nSample Prediction:")
print(f"Input: TV=$100k, Radio=$20k, Social Media=$30k")
print(f"\nLinear Regression:     {lr_model.predict(test_input)[0]:.2f}k units")
print(f"Polynomial Regression: {poly_model.predict(poly.transform(test_input))[0]:.2f}k units")
print(f"Random Forest:         {rf_model.predict(test_input)[0]:.2f}k units")
print(f"Gradient Boosting:     {gb_model.predict(test_input)[0]:.2f}k units (RECOMMENDED)")

## 8. Model Comparison & Summary

In [ ]:
# Create comparison table
comparison = pd.DataFrame({
    'Model': ['Linear', 'Polynomial', 'Random Forest', 'Gradient Boosting'],
    'Test R²': [lr_r2, poly_r2, rf_r2, gb_r2],
    'Test RMSE': [
        np.sqrt(mean_squared_error(y_test, y_pred_lr)),
        np.sqrt(mean_squared_error(y_test, y_pred_poly)),
        np.sqrt(mean_squared_error(y_test, y_pred_rf)),
        np.sqrt(mean_squared_error(y_test, y_pred_gb))
    ],
    'Test MAE': [
        mean_absolute_error(y_test, y_pred_lr),
        mean_absolute_error(y_test, y_pred_poly),
        mean_absolute_error(y_test, y_pred_rf),
        mean_absolute_error(y_test, y_pred_gb)
    ]
})

print("\n" + "="*80)
print("MODEL COMPARISON")
print("="*80)
print(comparison.to_string(index=False))
print("="*80)